# 🧬 Submission 2: Strict Mechanistic Generalization & Ensembling
**Strategy:** Pure Biophysics + Multi-Seed Ensembling

To guarantee out-of-distribution generalization for the Phase 2 wet-lab validation, this pipeline takes a highly defensive approach: we drop all ESM-2 embeddings to prevent evolutionary memorization, treat structural missingness as a biological signal, and ensemble across multiple random seeds for maximum stability.

In [1]:

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/competitions/retroviral-challenge-predict/family_splits.csv
/kaggle/input/competitions/retroviral-challenge-predict/sample_submission.csv
/kaggle/input/competitions/retroviral-challenge-predict/esm2_embeddings.npz
/kaggle/input/competitions/retroviral-challenge-predict/feature_dictionary.csv
/kaggle/input/competitions/retroviral-challenge-predict/train.csv
/kaggle/input/competitions/retroviral-challenge-predict/test.csv
/kaggle/input/competitions/retroviral-challenge-predict/structures/AVIRE-RT.pdb
/kaggle/input/competitions/retroviral-challenge-predict/structures/Gs-RT.pdb
/kaggle/input/competitions/retroviral-challenge-predict/structures/Drg-RT.pdb
/kaggle/input/competitions/retroviral-challenge-predict/structures/Ec48-RT.pdb
/kaggle/input/competitions/retroviral-challenge-predict/structures/Gate-RT.pdb
/kaggle/input/competitions/retroviral-challenge-predict/structures/Ne144-RT.pdb
/kaggle/input/competitions/retroviral-challenge-predict/structures/RSVSB-RT.pdb
/kaggle/i

In [2]:
import os
import numpy as np
import pandas as pd
import warnings
from scipy.stats import rankdata
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.svm import SVR
from sklearn.ensemble import VotingRegressor, VotingClassifier, ExtraTreesClassifier
from sklearn.feature_selection import SelectKBest, f_classif, f_regression
from sklearn.metrics import average_precision_score
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C

warnings.filterwarnings('ignore')

## 📐 1. Evaluation Metrics (CLS)
We implement the custom CLS metric. To perfectly align with the heavily penalized Weighted Spearman component, our continuous models will inject custom sample weights (`y_true + 0.01`) during training.

In [3]:
# ==========================================
# 1. EVALUATION METRICS (CLS)
# ==========================================
def calculate_weighted_spearman(y_true_eff: np.ndarray, y_pred: np.ndarray) -> float:
    weights = y_true_eff + 0.01
    rank_true = rankdata(y_true_eff)
    rank_pred = rankdata(y_pred)
    mean_true = np.average(rank_true, weights=weights)
    mean_pred = np.average(rank_pred, weights=weights)
    num = np.sum(weights * (rank_true - mean_true) * (rank_pred - mean_pred))
    den_true = np.sum(weights * (rank_true - mean_true)**2)
    den_pred = np.sum(weights * (rank_pred - mean_pred)**2)
    if den_true == 0 or den_pred == 0: return 0.0
    corr = num / np.sqrt(den_true * den_pred)
    return max(0.0, float(corr))

def calculate_cls(y_true_active, y_true_eff, y_pred):
    pr_auc = average_precision_score(y_true_active, y_pred)
    w_spearman = calculate_weighted_spearman(y_true_eff, y_pred)
    if pr_auc <= 0 or w_spearman <= 0: return 0.0, pr_auc, w_spearman
    cls_score = 2 * (pr_auc * w_spearman) / (pr_auc + w_spearman)
    return cls_score, pr_auc, w_spearman

## 🛠️ 2. Purely Mechanistic Feature Engineering
**Zero-Leakage Policy:** ESM-2 embeddings are permanently disabled, and leaky structural features (like `foldseek_TM_*`) are purged. We solely rely on bespoke local features (e.g., active-site quality) that do not reveal evolutionary lineage.

In [4]:
# ==========================================
# 2. FEATURE ENGINEERING (Strictly Mechanistic)
# ==========================================
def engineer_features(train_df, test_df):
    train_df = train_df.copy()
    test_df = test_df.copy()
    
    # 🧬 ONLY localized active-site features. No global features that leak length/family!
    if "triad_detected" in train_df.columns and "esmif_log_likelihood" in train_df.columns:
        for df in [train_df, test_df]:
            ll = df["esmif_log_likelihood"].fillna(df["esmif_log_likelihood"].median())
            df["feat_catalytic_fitness"] = df["triad_detected"].fillna(0) * np.exp(ll)

    if "triad_rmsd_to_hiv" in train_df.columns:
        for df in [train_df, test_df]:
            df["feat_active_site_quality"] = 1.0 / (1.0 + df["triad_rmsd_to_hiv"].fillna(99.0))

    return train_df, test_df

def load_competition_data(base_path):
    train_df = pd.read_csv(os.path.join(base_path, "train.csv"))
    test_df = pd.read_csv(os.path.join(base_path, "test.csv"))
    train_df.columns = train_df.columns.str.strip()
    test_df.columns = test_df.columns.str.strip()
    
    eff_col = next((c for c in train_df.columns if 'effic' in c.lower() or 'pct' in c.lower()), None)
    active_col = next((c for c in train_df.columns if 'active' in c.lower()), None)

    train_df, test_df = engineer_features(train_df, test_df)
    
    # 🔒 ESM is permanently disabled to ensure Phase 2 wet-lab survival
    esm_features = [] 
    
    leaky_prefixes = ['foldseek_TM_', 'rt_family', 'rt_name', 'sequence']
    handcrafted_features = [
        c for c in train_df.columns 
        if c not in [eff_col, active_col]
        and not any(str(c).startswith(leak) for leak in leaky_prefixes)
        and pd.api.types.is_numeric_dtype(train_df[c])
    ]
    return train_df, test_df, handcrafted_features, esm_features, eff_col, active_col


## ⚖️ 3. Metric-Aligned Custom Estimators
We wrap standard regressors (`Ridge`, `SVR`, `GaussianProcess`) into custom estimators that target `np.log1p(y)` and apply our Custom Rank sample weights. Crucially, we expose the `random_seed` parameter to enable robust variance reduction later.

In [5]:
# ==========================================
# 3. METRIC ALIGNED MODELS
# ==========================================
class LogMetricAlignedRidge(RegressorMixin, BaseEstimator):
    def __init__(self, alpha=15.0, random_seed=42): 
        self.alpha = alpha; self.random_seed = random_seed
    def fit(self, X, y, **fit_params):
        self.model = Ridge(alpha=self.alpha, random_state=self.random_seed)
        weights = np.asarray(y) + 0.01
        self.model.fit(X, np.log1p(y), sample_weight=weights)
        return self
    def predict(self, X): return np.maximum(0.0, np.expm1(self.model.predict(X)))

class LogMetricAlignedSVR(RegressorMixin, BaseEstimator):
    def __init__(self, C=2.0, epsilon=0.05): 
        self.C = C; self.epsilon = epsilon
    def fit(self, X, y, **fit_params):
        self.model = SVR(kernel='rbf', C=self.C, epsilon=self.epsilon)
        weights = np.asarray(y) + 0.01
        self.model.fit(X, np.log1p(y), sample_weight=weights)
        return self
    def predict(self, X): return np.maximum(0.0, np.expm1(self.model.predict(X)))

class LogMetricAlignedGP(RegressorMixin, BaseEstimator):
    def __init__(self, random_seed=42): 
        self.random_seed = random_seed
    def fit(self, X, y, **fit_params):
        weights = np.asarray(y) + 0.01
        noise_variance = 0.1 / weights 
        kernel = C(1.0, (1e-3, 1e3)) * Matern(length_scale=1.0, nu=1.5)
        self.model = GaussianProcessRegressor(kernel=kernel, alpha=noise_variance, n_restarts_optimizer=5, random_state=self.random_seed)
        self.model.fit(X, np.log1p(y))
        return self
    def predict(self, X): return np.maximum(0.0, np.expm1(self.model.predict(X)))


## 🧠 4. Decoupled Pipeline 
**The Golden Fix (`add_indicator=True`):** In structural biology, a `NaN` is rarely random noise. If ESMFold fails to resolve a motif, the enzyme is likely broken/inactive. By adding imputation indicators, we explicitly teach the model to learn from this "missingness."

Our final decoupled output logically sharpens biological confidence: $P(\text{active})^2 \times \text{Predicted\_Efficiency}$.

In [6]:
# ==========================================
# 4. DECOUPLED ARCHITECTURE
# ==========================================
def build_feature_pipeline(handcrafted_features, esm_features, task='regression', random_seed=42):
    transformers = []
    score_function = f_classif if task == 'classification' else f_regression
    k_feats = 10 if task == 'classification' else 15 
    
    hc_pipe = Pipeline([
        # THE FIX: Restoring add_indicator=True
        # This explicitly tells the models: "ESMFold failed to resolve this part of the protein!"
        ('imputer', SimpleImputer(strategy='median', add_indicator=True)), 
        ('scaler', StandardScaler()),
        ('selector', SelectKBest(score_func=score_function, k=k_feats)) 
    ])
    transformers.append(('handcrafted', hc_pipe, handcrafted_features))
        
    return ColumnTransformer(transformers)

class DecoupledExpectedValuePredictor(RegressorMixin, BaseEstimator):
    def __init__(self, classifier_pipe, regressor_pipe):
        self.classifier_pipe = classifier_pipe; self.regressor_pipe = regressor_pipe

    def fit(self, X, y, **fit_params):
        y_active = (np.asarray(y) > 0).astype(int)
        self.classifier_pipe_ = clone(self.classifier_pipe).fit(X, y_active)
        self.regressor_pipe_ = clone(self.regressor_pipe).fit(X, y, **fit_params)
        return self

    def predict(self, X):
        p_active = self.classifier_pipe_.predict_proba(X)[:, 1]
        pred_eff = self.regressor_pipe_.predict(X)
        return (p_active ** 2) * pred_eff # Biological Confidence Sharpening

def build_model_pipeline(handcrafted_features, esm_features, random_seed=42):
    clf_prep = build_feature_pipeline(handcrafted_features, esm_features, task='classification', random_seed=random_seed)
    clf1 = LogisticRegression(penalty='l1', solver='liblinear', C=0.2, class_weight='balanced', random_state=random_seed)
    clf2 = ExtraTreesClassifier(n_estimators=50, max_depth=2, class_weight='balanced', random_state=random_seed)
    gatekeeper_clf = VotingClassifier(estimators=[('lr', clf1), ('et', clf2)], voting='soft')
    clf_pipe = Pipeline([('prep', clf_prep), ('clf', gatekeeper_clf)])
    
    reg_prep = build_feature_pipeline(handcrafted_features, esm_features, task='regression', random_seed=random_seed)
    continuous_ranker = VotingRegressor(
        estimators=[
            ('ridge', LogMetricAlignedRidge(alpha=15.0, random_seed=random_seed)),
            ('svr', LogMetricAlignedSVR(C=2.0, epsilon=0.05)),
            ('gp', LogMetricAlignedGP(random_seed=random_seed))
        ], weights=[0.50, 0.20, 0.30] 
    )
    reg_pipe = Pipeline([('prep', reg_prep), ('reg', continuous_ranker)])
    return DecoupledExpectedValuePredictor(classifier_pipe=clf_pipe, regressor_pipe=reg_pipe)


## 🔄 5. Multi-Seed LOFO Validation & Smart Inference
Cross-validation on just 57 samples is inherently noisy. To combat this, we run our strict Leave-One-Family-Out (LOFO) validation across three random seeds (`[42, 1337, 2026]`) and average the out-of-fold predictions. 

This exact same seed-ensembling logic is dynamically applied to the final Phase 2 unseen test sequences, guaranteeing maximum stability in the Mandrake Bio wet-lab.

In [7]:

# ==========================================
# 5. LOFO HARNESS & SMART MAPPING
# ==========================================
def run_lofo_validation(df_train, handcrafted_feats, esm_feats, eff_col, active_col):
    print("\n--- Starting Leak-Free Leave-One-Family-Out (LOFO) Cross-Validation ---")
    families = df_train['rt_family'].unique()
    SEEDS = [42, 1337, 2026] 
    all_seed_predictions = []

    for seed in SEEDS:
        oof_predictions = np.zeros(len(df_train))
        for fold, family in enumerate(families):
            val_mask = df_train['rt_family'] == family
            train_mask = ~val_mask
            X_train, y_train = df_train[train_mask], df_train.loc[train_mask, eff_col]
            
            pipeline = build_model_pipeline(handcrafted_feats, esm_feats, random_seed=seed)
            pipeline.fit(X_train, y_train)
            oof_predictions[val_mask] = pipeline.predict(df_train[val_mask])
            
        all_seed_predictions.append(oof_predictions)
    
    final_oof_predictions = np.mean(all_seed_predictions, axis=0)
    y_true_active = df_train[active_col].values
    y_true_eff = df_train[eff_col].values
    cls_score, pr_auc, w_spearman = calculate_cls(y_true_active, y_true_eff, final_oof_predictions)
    
    print("\n" + "="*45)
    print("LOFO POOLED EVALUATION")
    print("="*45)
    print(f"PR-AUC (Classification)     : {pr_auc:.4f}")
    print(f"Weighted Spearman (Ranking) : {w_spearman:.4f}")
    print(f"Final CLS (Harmonic Mean)   : {cls_score:.4f}")
    print("="*45 + "\n")
    return final_oof_predictions

if __name__ == "__main__":
    KAGGLE_PATH = "/kaggle/input/competitions/retroviral-challenge-predict" 
    if not os.path.exists(KAGGLE_PATH): KAGGLE_PATH = "." 
        
    try:
        train_df, test_df, hc_feats, esm_feats, target_eff, target_act = load_competition_data(KAGGLE_PATH)
        oof_preds = run_lofo_validation(train_df, hc_feats, esm_feats, target_eff, target_act)
        
        print("Training Final Phase 2 Model on 100% available data...")
        SEEDS = [42, 1337, 2026]
        phase2_predictions = np.zeros(len(test_df))
        
        for seed in SEEDS:
            final_pipeline = build_model_pipeline(hc_feats, esm_feats, random_seed=seed)
            final_pipeline.fit(train_df, train_df[target_eff])
            phase2_predictions += final_pipeline.predict(test_df)
            
        phase2_predictions /= len(SEEDS)
        
        oof_dict = dict(zip(train_df['rt_name'], oof_preds))
        final_preds = [oof_dict.get(row['rt_name'], phase2_predictions[idx]) for idx, row in test_df.iterrows()]
                
        submission = pd.DataFrame({'rt_name': test_df['rt_name'], 'predicted_score': final_preds})
        submission.to_csv("submission.csv", index=False)
        print("✅ Created 'submission.csv'.")
        
    except FileNotFoundError:
        print("Dataset not found. Please ensure your path is correct if running locally!")


--- Starting Leak-Free Leave-One-Family-Out (LOFO) Cross-Validation ---

LOFO POOLED EVALUATION
PR-AUC (Classification)     : 0.6439
Weighted Spearman (Ranking) : 0.4041
Final CLS (Harmonic Mean)   : 0.4965

Training Final Phase 2 Model on 100% available data...
✅ Created 'submission.csv'.
